# 🏢 Rent Prediction Module

**Основная логика Module на FastAPI (в notebook формате)**

## 📌 Цель:
Принимает запрос с координатами → запрашивает данные у API и Yandex → получает прогноз от ML модели → возвращает результат.

## 🔗 Входные данные:
- lat, lon (координаты)
- square (площадь)
- property_type (тип недвижимости)
- дополнительные параметры

## 📤 Выходные данные:
- Прогнозируемая стоимость аренды
- Использованные признаки
- Время обработки

# Создание .env.example через Python (сделано)
env_content = """# Конфигурация базы данных
DB_HOST=localhost
DB_PORT=5432
DB_NAME=your_database_name
DB_USER=your_username
DB_PASSWORD=your_password

# Конфигурация MLFlow
MLFLOW_TRACKING_URI=http://localhost:5000
MLFLOW_EXPERIMENT_NAME=default

# Конфигурация API
API_HOST=0.0.0.0
API_PORT=8000
DEBUG=True
"""

with open('.env.example', 'w', encoding='utf-8') as f:
    f.write(env_content)

print("✅ Файл .env.example создан")

In [3]:
# ВСЁ в одной ячейке!
import asyncio
import pandas as pd
import numpy as np
import json
import uuid
from datetime import datetime
import logging
from typing import Dict, List, Optional, Any

# Настройка логирования
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Импорты загружены")

✅ Импорты загружены


In [4]:
# Data classes для запросов и ответов
from dataclasses import dataclass, asdict

@dataclass
class RentRequest:
    """Запрос на прогноз аренды"""
    lat: float
    lon: float
    square: float
    property_type: str = "office"
    floor: Optional[int] = None
    year_built: Optional[int] = None
    radius: int = 500
    
    def to_dict(self):
        return asdict(self)

@dataclass
class ApiResponse:
    """Ответ от внешнего API"""
    metro_distance: int
    poi_count: int
    traffic_level: str
    business_centers: int

@dataclass
class YandexResponse:
    """Ответ от Yandex API"""
    address: str
    district: str
    rubrics: List[str]
    metro_name: Optional[str]
    population_density: int

@dataclass
class PredictionResult:
    """Итоговый результат"""
    predicted_price: float
    currency: str
    period: str
    request_id: str
    processing_time_ms: float
    features_used: List[str]
    timestamp: str

print("✅ Модели данных созданы")

✅ Модели данных созданы


In [5]:
async def process_rent_prediction(request: RentRequest) -> Dict[str, Any]:
    """
    ОСНОВНАЯ ФУНКЦИЯ MODULE
    """
    start_time = datetime.now()
    request_id = f"req_{uuid.uuid4().hex[:8]}"
    
    logger.info(f"🔄 Начало обработки {request_id}")
    logger.info(f"📥 Запрос: {request.to_dict()}")
    
    try:
        # 1. Имитация параллельных запросов к API
        logger.info(f"🌐 Запрос данных от API и Yandex...")
        
        async def call_external_api():
            """Имитация API карт"""
            await asyncio.sleep(0.4)  # Задержка сети
            return {
                "metro_distance": np.random.randint(100, 2000),
                "poi_count": np.random.randint(10, 100),
                "traffic_level": np.random.choice(["low", "medium", "high"]),
                "business_centers": np.random.randint(0, 10),
                "parking_spots": np.random.randint(0, 50)
            }
        
        async def call_yandex_api():
            """Имитация Yandex API"""
            await asyncio.sleep(0.5)
            return {
                "address": f"Москва, {request.lat:.4f}, {request.lon:.4f}",
                "district": np.random.choice(["ЦАО", "САО", "ЮЗАО", "ЗАО", "ВАО"]),
                "rubrics": ["business", "office", "commercial"],
                "metro_name": np.random.choice(["Охотный ряд", "Деловой центр", "Пушкинская", None]),
                "population_density": np.random.randint(5000, 25000),
                "business_score": round(np.random.uniform(6.0, 9.5), 1)
            }
        
        # ПАРАЛЛЕЛЬНЫЕ ВЫЗОВЫ (как в примере с await asyncio.gather)
        api_task = call_external_api()
        yandex_task = call_yandex_api()
        
        api_data, yandex_data = await asyncio.gather(api_task, yandex_task)
        
        # 2. Подготовка признаков для ML модели
        features = {
            "square": request.square,
            "property_type": request.property_type,
            "floor": request.floor or 1,
            "year_built": request.year_built or 2000,
            "metro_distance": api_data["metro_distance"],
            "poi_count": api_data["poi_count"],
            "district": yandex_data["district"],
            "traffic_level": api_data["traffic_level"],
            "business_score": yandex_data.get("business_score", 7.0),
            "has_metro": yandex_data["metro_name"] is not None
        }
        
        logger.info(f"📊 Признаки для ML: {len(features)} параметров")
        
        # 3. Имитация запроса к MLFlow модели
        logger.info(f"🤖 Запрос к ML модели...")
        await asyncio.sleep(0.3)
        
        # Простая формула для демо
        base_price = request.square * 1500  # руб/м²
        
        # Модификаторы
        type_modifiers = {
            "office": 1.0,
            "retail": 1.8,
            "warehouse": 0.6,
            "cafe": 1.3,
            "restaurant": 1.5
        }
        
        metro_modifier = 1.3 if api_data["metro_distance"] < 500 else 1.0
        type_modifier = type_modifiers.get(request.property_type, 1.0)
        traffic_modifier = 1.2 if api_data["traffic_level"] == "high" else 1.0
        
        predicted_price = base_price * type_modifier * metro_modifier * traffic_modifier
        
        # Добавляем случайность
        predicted_price *= np.random.uniform(0.95, 1.05)
        
        # 4. Формирование результата
        processing_time = (datetime.now() - start_time).total_seconds() * 1000
        
        result = PredictionResult(
            predicted_price=round(predicted_price, 2),
            currency="RUB",
            period="month",
            request_id=request_id,
            processing_time_ms=round(processing_time, 2),
            features_used=list(features.keys()),
            timestamp=datetime.now().isoformat()
        )
        
        logger.info(f"✅ Завершено за {result.processing_time_ms} мс")
        logger.info(f"💰 Прогноз: {result.predicted_price} RUB/месяц")
        
        # 5. Возвращаем структурированный ответ
        return {
            "success": True,
            "data": asdict(result),
            "source_data": {
                "api_response": api_data,
                "yandex_response": yandex_data,
                "ml_features": features
            },
            "metadata": {
                "module_version": "1.0.0",
                "processing_steps": ["api_call", "yandex_call", "ml_prediction"]
            }
        }
        
    except Exception as e:
        logger.error(f"❌ Ошибка: {str(e)}")
        return {
            "success": False,
            "error": str(e),
            "request_id": request_id,
            "timestamp": datetime.now().isoformat()
        }

print("✅ Основная функция Module создана!")

✅ Основная функция Module создана!


In [6]:
# Тестируем нашу функцию
async def run_test():
    """Тестовый запрос"""
    print("🧪 ТЕСТИРОВАНИЕ MODULE")
    print("=" * 50)
    
    test_request = RentRequest(
        lat=55.7558,
        lon=37.6173,
        square=100,
        property_type="office",
        floor=5,
        year_built=2015
    )
    
    print(f"📥 Тестовый запрос:")
    for key, value in test_request.to_dict().items():
        print(f"  {key}: {value}")
    
    print(f"\\n🔄 Обработка...")
    result = await process_rent_prediction(test_request)
    
    print(f"\\n📤 Результат:")
    if result["success"]:
        data = result["data"]
        print(f"  ✅ Успешно!")
        print(f"  💰 Цена: {data['predicted_price']} {data['currency']}/{data['period']}")
        print(f"  ⏱️  Время: {data['processing_time_ms']} мс")
        print(f"  🆔 ID: {data['request_id']}")
        print(f"  📊 Признаков: {len(data['features_used'])}")
        
        # Показываем часть данных от API
        print(f"\\n  📍 Данные от API:")
        print(f"    Метро: {result['source_data']['api_response']['metro_distance']} м")
        print(f"    POI: {result['source_data']['api_response']['poi_count']}")
        print(f"    Район: {result['source_data']['yandex_response']['district']}")
    else:
        print(f"  ❌ Ошибка: {result['error']}")
    
    return result

# Запускаем тест
test_result = await run_test()

2025-12-24 11:04:44,569 - 🔄 Начало обработки req_84e35e57
2025-12-24 11:04:44,569 - 📥 Запрос: {'lat': 55.7558, 'lon': 37.6173, 'square': 100, 'property_type': 'office', 'floor': 5, 'year_built': 2015, 'radius': 500}
2025-12-24 11:04:44,570 - 🌐 Запрос данных от API и Yandex...


🧪 ТЕСТИРОВАНИЕ MODULE
📥 Тестовый запрос:
  lat: 55.7558
  lon: 37.6173
  square: 100
  property_type: office
  floor: 5
  year_built: 2015
  radius: 500
\n🔄 Обработка...


2025-12-24 11:04:45,093 - 📊 Признаки для ML: 10 параметров
2025-12-24 11:04:45,094 - 🤖 Запрос к ML модели...
2025-12-24 11:04:45,400 - ✅ Завершено за 831.82 мс
2025-12-24 11:04:45,402 - 💰 Прогноз: 147883.63 RUB/месяц


\n📤 Результат:
  ✅ Успешно!
  💰 Цена: 147883.63 RUB/month
  ⏱️  Время: 831.82 мс
  🆔 ID: req_84e35e57
  📊 Признаков: 10
\n  📍 Данные от API:
    Метро: 1976 м
    POI: 40
    Район: ВАО


In [7]:
# Экспортируем функцию для использования в других ноутбуках
import pickle

# Сохраняем нашу функцию
with open('module_function.pkl', 'wb') as f:
    pickle.dump(process_rent_prediction, f)

print("✅ Функция сохранена в module_function.pkl")

# Можно также сохранить как JSON для документации
module_info = {
    "name": "RentPredictionModule",
    "version": "1.0.0",
    "description": "Модуль прогноза стоимости аренды коммерческой недвижимости",
    "inputs": ["lat", "lon", "square", "property_type", "floor", "year_built", "radius"],
    "outputs": ["predicted_price", "features_used", "processing_time"],
    "dependencies": ["asyncio", "pandas", "numpy"]
}

with open('module_info.json', 'w', encoding='utf-8') as f:
    json.dump(module_info, f, indent=2, ensure_ascii=False)

print("✅ Информация о модуле сохранена в module_info.json")

✅ Функция сохранена в module_function.pkl
✅ Информация о модуле сохранена в module_info.json
